# SpecWise — RAG Pipeline Notebook

In [4]:
import os
import re
import json
from pathlib import Path

import chromadb
from sentence_transformers import SentenceTransformer

try:
    from pypdf import PdfReader
    PYPDF_AVAILABLE = True
except ImportError:
    PYPDF_AVAILABLE = False

DOCS_DIR = Path("../data/raw_docs/library-management-system")
VECTOR_STORE_DIR = Path("../backend/data/vector_store")
PROJECT_ID = "library_management_demo"          
COLLECTION_NAME = f"project_{PROJECT_ID}"

CHUNK_SIZE_WORDS = 400     
CHUNK_OVERLAP_WORDS = 60
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"

print("Docs dir exists:", DOCS_DIR.exists())
print("Vector store dir:", VECTOR_STORE_DIR.resolve())


Docs dir exists: True
Vector store dir: D:\ITI AI level2\1. Final Project\SpecWise-RAG\backend\data\vector_store


## 2.1 Load & Inspect

We load every document under `data/sample_project/`, report basic stats per file (format, size, word count), and flag anything that fails to parse (e.g. a scanned PDF with no extractable text layer).

In [5]:
def load_text_file(path: Path) -> str:
    return path.read_text(encoding="utf-8")


def load_pdf_file(path: Path) -> str:
    if not PYPDF_AVAILABLE:
        raise RuntimeError("pypdf not installed — run `uv add pypdf`")
    reader = PdfReader(str(path))
    text_parts = [page.extract_text() or "" for page in reader.pages]
    return "\n".join(text_parts)


def load_document(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix in (".md", ".txt"):
        return load_text_file(path)
    elif suffix == ".pdf":
        return load_pdf_file(path)
    else:
        raise ValueError(f"Unsupported file type: {suffix}")


raw_documents = {}   # doc_name -> raw text
load_report = []      # one row per file, for the markdown report below

for path in sorted(DOCS_DIR.glob("*")):
    if path.is_dir():
        continue
    row = {"file": path.name, "format": path.suffix.lower(), "status": "ok", "note": ""}
    try:
        text = load_document(path)
        if not text.strip():
            row["status"] = "empty"
            row["note"] = "No extractable text — likely a scanned/image-only file, needs OCR."
        else:
            raw_documents[path.name] = text
            row["note"] = f"{len(text.split())} words"
    except Exception as e:
        row["status"] = "failed"
        row["note"] = str(e)
    load_report.append(row)

for row in load_report:
    print(f"[{row['status'].upper():6}] {row['file']:20} {row['format']:6} {row['note']}")

print(f"\nLoaded {len(raw_documents)} of {len(load_report)} files successfully.")


[OK    ] API_Spec.md          .md    250 words
[OK    ] SRS.md               .md    360 words
[OK    ] UseCases.md          .md    369 words

Loaded 3 of 3 files successfully.


## 2.2 Chunking Strategy

In [6]:
HEADER_PATTERN = re.compile(r"^(#{1,3})\s+(.*)$", re.MULTILINE)


def split_into_sections(text: str):
    """Split a markdown doc into (heading, section_text) pairs using # / ## / ### headers.
    Falls back to treating the whole doc as one section if no headers are found.
    """
    matches = list(HEADER_PATTERN.finditer(text))
    if not matches:
        return [(None, text.strip())]
    sections = []
    for i, m in enumerate(matches):
        heading = m.group(2).strip()
        start = m.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        body = text[start:end].strip()
        if body:
            sections.append((heading, body))
    return sections


def sliding_word_windows(words, size, overlap):
    """Slide a window of `size` words over `words`, stepping by (size - overlap)."""
    if len(words) <= size:
        return [words]
    windows = []
    step = size - overlap
    for start in range(0, len(words), step):
        window = words[start:start + size]
        if not window:
            break
        windows.append(window)
        if start + size >= len(words):
            break
    return windows


def chunk_document(doc_name: str, text: str):
    chunks = []
    for heading, body in split_into_sections(text):
        words = body.split()
        if len(words) <= CHUNK_SIZE_WORDS:
            chunks.append({"doc_name": doc_name, "section": heading, "text": body, "n_words": len(words)})
        else:
            for window in sliding_word_windows(words, CHUNK_SIZE_WORDS, CHUNK_OVERLAP_WORDS):
                chunks.append({"doc_name": doc_name, "section": heading, "text": " ".join(window), "n_words": len(window)})
    return chunks


all_chunks = []
for doc_name, text in raw_documents.items():
    doc_chunks = chunk_document(doc_name, text)
    all_chunks.extend(doc_chunks)
    print(f"{doc_name}: {len(doc_chunks)} chunks")

# stable chunk ids: <doc_name>::<index>
for i, c in enumerate(all_chunks):
    c["chunk_id"] = f"{c['doc_name']}::{i}"

print(f"\nTotal chunks: {len(all_chunks)}")
print(f"Avg words/chunk: {sum(c['n_words'] for c in all_chunks) / len(all_chunks):.1f}")
print(f"Max words/chunk: {max(c['n_words'] for c in all_chunks)}")
print(f"Min words/chunk: {min(c['n_words'] for c in all_chunks)}")


API_Spec.md: 6 chunks
SRS.md: 6 chunks
UseCases.md: 6 chunks

Total chunks: 18
Avg words/chunk: 49.0
Max words/chunk: 72
Min words/chunk: 29


In [7]:
# Inspect one chunk to sanity-check boundaries line up with a real requirement
sample = next(c for c in all_chunks if c["section"] and "Reservation" in c["section"])
print(f"doc = {sample['doc_name']}")
print(f"section = {sample['section']}")
print(f"chunk_id = {sample['chunk_id']}")
print("---")
print(sample["text"])


doc = SRS.md
section = 3.4 Reservation Rules
chunk_id = SRS.md::9
---
If a book is currently checked out, a member may place a reservation on it. Yes, a member may reserve a book that is already checked out; the system shall notify the reserving member by email within 24 hours of the book's return. Reservations expire automatically after 3 days if the reserved book is not picked up. A member may hold no more than 3 active reservations at once.


## 2.3 Embeddings & Vector Store

In [8]:
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

texts = [c["text"] for c in all_chunks]
embeddings = embedding_model.encode(texts, show_progress_bar=True).tolist()

print(f"Embedded {len(embeddings)} chunks, dimension = {len(embeddings[0])}")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedded 18 chunks, dimension = 384


In [9]:
VECTOR_STORE_DIR.mkdir(parents=True, exist_ok=True)

chroma_client = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR))
collection = chroma_client.get_or_create_collection(name=COLLECTION_NAME)

collection.add(
    ids=[c["chunk_id"] for c in all_chunks],
    embeddings=embeddings,
    documents=[c["text"] for c in all_chunks],
    metadatas=[{"doc_name": c["doc_name"], "section": c["section"] or ""} for c in all_chunks],
)

print(f"Collection '{COLLECTION_NAME}' now has {collection.count()} chunks persisted at {VECTOR_STORE_DIR.resolve()}")


Collection 'project_library_management_demo' now has 18 chunks persisted at D:\ITI AI level2\1. Final Project\SpecWise-RAG\backend\data\vector_store


In [10]:
# Fresh client + fresh collection handle, no re-use of the objects created above
fresh_client = chromadb.PersistentClient(path=str(VECTOR_STORE_DIR))
fresh_collection = fresh_client.get_collection(name=COLLECTION_NAME)
print(f"Reloaded collection count: {fresh_collection.count()}")

test_question = "Can a member reserve a book that's already checked out?"
query_embedding = embedding_model.encode([test_question]).tolist()

results = fresh_collection.query(query_embeddings=query_embedding, n_results=3)

for rank, (doc, meta, dist) in enumerate(zip(results["documents"][0], results["metadatas"][0], results["distances"][0]), start=1):
    print(f"\n#{rank}  distance={dist:.4f}  doc={meta['doc_name']}  section={meta['section']}")
    print(doc[:220])


Reloaded collection count: 18

#1  distance=0.5087  doc=SRS.md  section=3.4 Reservation Rules
If a book is currently checked out, a member may place a reservation on it. Yes, a member may reserve a book that is already checked out; the system shall notify the reserving member by email within 24 hours of the book'

#2  distance=0.8123  doc=UseCases.md  section=UC-3: Borrow a Book
Actor: Member.
Preconditions: Member is in good standing and has fewer than 5 items currently checked out.
Main flow: Member presents the book and library card at checkout (or scans both at a self-checkout kiosk). The sy

#3  distance=0.8981  doc=API_Spec.md  section=POST /reservations
Places a reservation on a book that is currently checked out.
Request body: member_id (string), book_id (string).
Response: 201 Created with the reservation's queue position, or 400 Bad Request if the member already hold


## Section 2.4 (Retrieval & Prompting)

In [11]:
import ollama

OLLAMA_MODEL = "llama3.2"         
TOP_K = 4                           
DISTANCE_THRESHOLD = 0.75 


In [12]:
def retrieve(question: str, k: int = TOP_K):
    q_embedding = embedding_model.encode([question]).tolist()
    results = fresh_collection.query(query_embeddings=q_embedding, n_results=k)
    hits = []
    for doc, meta, dist, chunk_id in zip(
        results["documents"][0], results["metadatas"][0], results["distances"][0], results["ids"][0]
    ):
        hits.append({
            "chunk_id": chunk_id,
            "doc_name": meta["doc_name"],
            "section": meta["section"],
            "text": doc,
            "distance": dist,
        })
    return hits


def is_grounded_enough(hits) -> bool:
    """Grounded-refusal gate: if even the best match is too far away, don't bother calling the LLM."""
    if not hits:
        return False
    return hits[0]["distance"] <= DISTANCE_THRESHOLD


REFUSAL_MESSAGE_TEMPLATE = "This isn't covered in {project}'s uploaded documents."


In [13]:
def build_prompt(question: str, hits) -> str:
    context_blocks = []
    for h in hits:
        context_blocks.append(f"[Source: {h['doc_name']}, section: {h['section']}]\n{h['text']}")
    context = "\n\n".join(context_blocks)

    return f"""You are a documentation assistant. Answer the question using ONLY the context below.
If the answer is not contained in the context, say explicitly that it isn't covered in the documents -
do not guess or use outside knowledge. When you do answer, cite the source document and section for
every fact you use, in the form (doc_name, section).

CONTEXT:
{context}

QUESTION: {question}

ANSWER:"""


def generate_answer(question: str, project_label: str = "this project"):
    """Full pipeline: retrieve -> grounded-refusal gate -> (maybe) generate -> answer + sources."""
    hits = retrieve(question)

    if not is_grounded_enough(hits):
        return {
            "answer": REFUSAL_MESSAGE_TEMPLATE.format(project=project_label),
            "sources": [],
            "refused": True,
            "top_distance": hits[0]["distance"] if hits else None,
        }

    prompt = build_prompt(question, hits)
    try:
        response = ollama.chat(model=OLLAMA_MODEL, messages=[{"role": "user", "content": prompt}])
        answer_text = response["message"]["content"]
    except Exception as e:
        answer_text = f"[Ollama call failed: {e}. Is `ollama serve` running and is '{OLLAMA_MODEL}' pulled?]"

    return {
        "answer": answer_text,
        "sources": [{"doc_name": h["doc_name"], "section": h["section"]} for h in hits],
        "refused": False,
        "top_distance": hits[0]["distance"],
    }


## Manual test — 10+ questions across the plan's evaluation categories

In [14]:
test_questions = [
    # Direct factual
    ("What is the maximum file upload size for staff document attachments?", "direct_factual"),
    ("How many days is the standard loan period for adult members?", "direct_factual"),
    ("What is charged for a lost or damaged library item?", "direct_factual"),
    # Conceptual
    ("Why does the system block borrowing once a member's fines exceed $10?", "conceptual"),
    ("What happens if a reserved book isn't picked up in time?", "conceptual"),
    # Multi-document
    ("Does the use case for reserving a book match what the SRS says about reservation limits?", "multi_document"),
    ("Is the borrowing rule in the SRS consistent with what the loans API endpoint enforces?", "multi_document"),
    # Difficult / paraphrased retrieval
    ("Can someone put a hold on a book somebody else currently has checked out?", "difficult_retrieval"),
    ("What stops a person from grabbing more books than they're allowed to have at once?", "difficult_retrieval"),
    # Unanswerable (not covered by these documents)
    ("What is the total marketing budget allocated for this library project?", "unanswerable"),
    ("Who is the current mayor of the city where this library is located?", "unanswerable"),
    ("What programming language is the frontend written in?", "unanswerable"),
]

print(f"{len(test_questions)} test questions queued.\n")

manual_test_log = []
for question, category in test_questions:
    result = generate_answer(question, project_label="Library Management System")
    manual_test_log.append({"question": question, "category": category, **result})

    print(f"[{category:18}] {question}")
    print(f"   top_distance={result['top_distance']}  refused={result['refused']}")
    if result["sources"]:
        print(f"   sources={result['sources']}")
    print(f"   answer: {result['answer'][:200]}")
    print()


12 test questions queued.

[direct_factual    ] What is the maximum file upload size for staff document attachments?
   top_distance=0.15464568138122559  refused=False
   sources=[{'doc_name': 'API_Spec.md', 'section': 'File Upload Limits'}, {'doc_name': 'SRS.md', 'section': '3.3 Borrowing Rules'}, {'doc_name': 'API_Spec.md', 'section': 'POST /loans'}, {'doc_name': 'SRS.md', 'section': '3.1 User Registration'}]
   answer: [Ollama call failed: Failed to connect to Ollama. Please check that Ollama is downloaded, running and accessible. https://ollama.com/download. Is `ollama serve` running and is 'llama3.2' pulled?]

[direct_factual    ] How many days is the standard loan period for adult members?
   top_distance=0.5298128128051758  refused=False
   sources=[{'doc_name': 'SRS.md', 'section': '3.3 Borrowing Rules'}, {'doc_name': 'API_Spec.md', 'section': 'POST /loans'}, {'doc_name': 'SRS.md', 'section': '3.1 User Registration'}, {'doc_name': 'UseCases.md', 'section': 'UC-3: Borrow a Book'

## Section 2.6 (Evaluation)

In [15]:
import pandas as pd

eval_rows = []
for question, category in test_questions:
    result = generate_answer(question, project_label="Library Management System")
    hits = retrieve(question)
    top_hit = hits[0] if hits else None

    eval_rows.append({
        "question": question,
        "category": category,
        "top_doc": top_hit["doc_name"] if top_hit else None,
        "top_section": top_hit["section"] if top_hit else None,
        "top_distance": round(result["top_distance"], 3) if result["top_distance"] is not None else None,
        "refused": result["refused"],
        "answer": result["answer"],
        # Fill these in by hand after reading the `answer` column above:
        "context_relevant": None,   # yes / no
        "grounded": None,           # yes / no / n/a (n/a if refused — nothing to ground)
        "failure_stage": None,      # blank if correct; else "retrieval" or "generation"
    })

eval_df = pd.DataFrame(eval_rows)
eval_df


,question,category,top_doc,top_section,top_distance,refused,answer,context_relevant,grounded,failure_stage
0,What is the maximum file upload size for staff...,direct_factual,API_Spec.md,File Upload Limits,0.155,False,[Ollama call failed: Failed to connect to Olla...,None,None,None
1,How many days is the standard loan period for ...,direct_factual,SRS.md,3.3 Borrowing Rules,0.530,False,[Ollama call failed: Failed to connect to Olla...,None,None,None
2,What is charged for a lost or damaged library ...,direct_factual,SRS.md,3.5 Fines and Penalties,0.880,True,This isn't covered in Library Management Syste...,None,None,None
3,Why does the system block borrowing once a mem...,conceptual,API_Spec.md,POST /loans,1.021,True,This isn't covered in Library Management Syste...,None,None,None
4,What happens if a reserved book isn't picked u...,conceptual,SRS.md,3.4 Reservation Rules,0.717,False,[Ollama call failed: Failed to connect to Olla...,None,None,None
5,Does the use case for reserving a book match w...,multi_document,SRS.md,3.4 Reservation Rules,0.738,False,[Ollama call failed: Failed to connect to Olla...,None,None,None
6,Is the borrowing rule in the SRS consistent wi...,multi_document,API_Spec.md,POST /loans,1.106,True,This isn't covered in Library Management Syste...,None,None,None
7,Can someone put a hold on a book somebody else...,difficult_retrieval,SRS.md,3.4 Reservation Rules,0.770,True,This isn't covered in Library Management Syste...,None,None,None
8,What stops a person from grabbing more books t...,difficult_retrieval,SRS.md,3.5 Fines and Penalties,1.004,True,This isn't covered in Library Management Syste...,None,None,None
9,What is the total marketing budget allocated f...,unanswerable,API_Spec.md,File Upload Limits,1.467,True,This isn't covered in Library Management Syste...,None,None,None


In [16]:
EVAL_EXPORT_PATH = Path("../docs/evaluation_results.csv")
EVAL_EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
eval_df.to_csv(EVAL_EXPORT_PATH, index=False)
print(f"Saved to {EVAL_EXPORT_PATH.resolve()}")

# Quick summary stats — recompute after filling in judgments
if eval_df["context_relevant"].notna().any():
    print()
    print("Retrieval relevant:", (eval_df["context_relevant"] == "yes").sum(), "/", len(eval_df))
    print("Answers grounded:  ", (eval_df["grounded"] == "yes").sum(), "/", (eval_df["grounded"] != "n/a").sum())
else:
    print("\n(Judgment columns not filled in yet — summary stats will be 0 until you do.)")


Saved to D:\ITI AI level2\1. Final Project\SpecWise-RAG\docs\evaluation_results.csv

(Judgment columns not filled in yet — summary stats will be 0 until you do.)
